<a href="https://colab.research.google.com/github/sruthims1/thinkpalm-agentai-SruthiMS-React-Agent/blob/main/react_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReAct Agent — Google Colab

| Field  | Details |
|--------|---------|
| Name   | Sruthi M S |
| Lab    | ReAct Agent Lab |

A **ReAct (Reasoning + Acting)** agent that interleaves step-by-step reasoning with real tool calls before producing a grounded final answer.

## Step 1 — Install dependencies

In [1]:
!pip install -q anthropic requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 18.3 MB/s eta 0:00:00


## Step 2 — Set your Anthropic API key

Either paste your key below **or** add it as a Colab Secret named `ANTHROPIC_API_KEY` (recommended).

In [2]:
import os

# Option A: Colab Secrets (recommended — click the key icon in the left sidebar)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    # Option B: paste directly (not recommended for shared notebooks)
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-your-key-here"
    print("API key set manually.")

API key loaded from Colab Secrets.


## Step 3 — Define the tools (7 total)

In [3]:
import math
import datetime
import subprocess
import sys
import json
from urllib.parse import quote
import requests

# ── Knowledge base ─────────────────────────────────────────────────────────────
KNOWLEDGE_BASE = {
    "python": "Python is a high-level, interpreted programming language prized for its readability and versatility.",
    "anthropic": "Anthropic is an AI safety company founded in 2021 that created the Claude family of large language models.",
    "claude": "Claude is a family of large language models developed by Anthropic, designed to be helpful, harmless, and honest.",
    "react": "ReAct (Reasoning + Acting) is an AI agent paradigm that interleaves step-by-step reasoning (Thought) with tool invocations (Action) and their results (Observation).",
    "llm": "A Large Language Model (LLM) is a deep-learning model trained on massive text corpora that can generate, summarise, and reason about natural language.",
    "agent": "An AI agent perceives its environment, reasons about goals, and takes actions via tools to accomplish tasks autonomously.",
    "tool use": "Tool use lets an LLM invoke external functions or APIs during generation, grounding responses in real data or computation.",
    "agentic ai": "Agentic AI refers to systems that operate autonomously over multiple steps using tools and memory to complete complex tasks.",
    "transformer": "The Transformer is a neural network architecture (2017) using self-attention, forming the foundation of nearly all modern LLMs.",
    "rag": "Retrieval-Augmented Generation combines an LLM with a retrieval system to fetch relevant documents before generating an answer.",
    "fine-tuning": "Fine-tuning adapts a pre-trained model to a specific task by continuing training on a smaller, curated dataset.",
    "prompt engineering": "Prompt engineering crafts inputs to language models to elicit desired outputs using techniques like chain-of-thought and few-shot examples.",
}

# ── Tool implementations ───────────────────────────────────────────────────────
def calculator(expression: str) -> str:
    safe_ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe_ns["__builtins__"] = {}
    try:
        return str(eval(expression, safe_ns))
    except ZeroDivisionError:
        return "Error: division by zero."
    except Exception as exc:
        return f"Error evaluating '{expression}': {exc}"

def get_current_time() -> str:
    return datetime.datetime.now().strftime("%A, %B %d, %Y at %I:%M:%S %p")

def lookup(term: str) -> str:
    key = term.strip().lower()
    if key in KNOWLEDGE_BASE:
        return KNOWLEDGE_BASE[key]
    for kb_key, kb_val in KNOWLEDGE_BASE.items():
        if key in kb_key or kb_key in key:
            return kb_val
    return f"Term '{term}' not found. Available: {', '.join(KNOWLEDGE_BASE)}"

def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    f, t = from_unit.strip().lower(), to_unit.strip().lower()
    def _to_c(v, u):
        if u == "celsius": return v
        if u == "fahrenheit": return (v - 32) * 5 / 9
        if u == "kelvin": return v - 273.15
    def _from_c(v, u):
        if u == "celsius": return v
        if u == "fahrenheit": return v * 9 / 5 + 32
        if u == "kelvin": return v + 273.15
    if f in {"celsius","fahrenheit","kelvin"} and t in {"celsius","fahrenheit","kelvin"}:
        return f"{value} {from_unit} = {_from_c(_to_c(value, f), t):.4f} {to_unit}"
    dist = {"meters":1,"m":1,"km":1000,"miles":1609.344,"feet":0.3048,"ft":0.3048}
    if f in dist and t in dist:
        return f"{value} {from_unit} = {value * dist[f] / dist[t]:.4f} {to_unit}"
    weight = {"grams":1,"g":1,"kg":1000,"pounds":453.592,"lb":453.592,"ounces":28.3495,"oz":28.3495}
    if f in weight and t in weight:
        return f"{value} {from_unit} = {value * weight[f] / weight[t]:.4f} {to_unit}"
    return f"Unsupported conversion: '{from_unit}' → '{to_unit}'."

def web_search(query: str) -> str:
    try:
        resp = requests.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": "1", "skip_disambig": "1"},
            headers={"User-Agent": "ReActAgent/2.0"}, timeout=10,
        )
        data = resp.json()
        parts = []
        if data.get("AbstractText"): parts.append(f"{data['AbstractText']} (Source: {data.get('AbstractSource','')})")
        if data.get("Answer"): parts.append(f"Quick answer: {data['Answer']}")
        topics = [f"• {t['Text']}" for t in data.get("RelatedTopics",[])[:3] if isinstance(t,dict) and t.get("Text")]
        if topics: parts.append("Related:\n" + "\n".join(topics))
        return "\n\n".join(parts) if parts else f"No instant answer for '{query}'. Try the lookup tool for AI/tech concepts."
    except Exception as exc:
        return f"Web search error: {exc}"

def get_weather(location: str) -> str:
    try:
        resp = requests.get(f"https://wttr.in/{quote(location)}?format=j1",
                            headers={"User-Agent": "ReActAgent/2.0"}, timeout=10)
        data = resp.json()
        cur = data["current_condition"][0]
        area = data["nearest_area"][0]
        return (f"Weather for {area['areaName'][0]['value']}, {area['country'][0]['value']}:\n"
                f"  Condition  : {cur['weatherDesc'][0]['value']}\n"
                f"  Temp       : {cur['temp_C']}°C / {cur['temp_F']}°F\n"
                f"  Humidity   : {cur['humidity']}%\n"
                f"  Wind       : {cur['windspeedKmph']} km/h")
    except Exception as exc:
        return f"Error fetching weather for '{location}': {exc}"

def run_python(code: str) -> str:
    try:
        result = subprocess.run([sys.executable, "-c", code],
                                capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            return f"RuntimeError:\n{result.stderr.strip()}"
        return result.stdout.strip() or "(ran successfully — no output)"
    except subprocess.TimeoutExpired:
        return "Error: execution timed out (10s)."
    except Exception as exc:
        return f"Error: {exc}"

# ── Dispatcher ─────────────────────────────────────────────────────────────────
def execute_tool(name: str, inputs: dict) -> str:
    dispatch = {
        "calculator":       lambda i: calculator(i["expression"]),
        "get_current_time": lambda _: get_current_time(),
        "lookup":           lambda i: lookup(i["term"]),
        "unit_converter":   lambda i: unit_converter(i["value"], i["from_unit"], i["to_unit"]),
        "web_search":       lambda i: web_search(i["query"]),
        "get_weather":      lambda i: get_weather(i["location"]),
        "run_python":       lambda i: run_python(i["code"]),
    }
    handler = dispatch.get(name)
    if handler is None:
        return f"Unknown tool '{name}'."
    try:
        return handler(inputs)
    except KeyError as exc:
        return f"Missing parameter for '{name}': {exc}"
    except Exception as exc:
        return f"Tool '{name}' error: {exc}"

# ── Tool schemas ───────────────────────────────────────────────────────────────
TOOL_SCHEMAS = [
    {"name": "calculator",
     "description": "Evaluates a math expression. Supports all Python arithmetic and math module functions (sqrt, log, sin, cos, etc.).",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string", "description": "Math expression, e.g. 'sqrt(256) + 2**10'"}}, "required": ["expression"]}},
    {"name": "get_current_time",
     "description": "Returns the current local date and time.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "lookup",
     "description": "Looks up an AI/tech concept. Terms: python, anthropic, claude, react, llm, agent, tool use, agentic ai, transformer, rag, fine-tuning, prompt engineering.",
     "input_schema": {"type": "object", "properties": {"term": {"type": "string"}}, "required": ["term"]}},
    {"name": "unit_converter",
     "description": "Converts between units. Temperature: celsius/fahrenheit/kelvin. Distance: km/miles/meters/feet. Weight: kg/pounds/grams/ounces.",
     "input_schema": {"type": "object", "properties": {
         "value": {"type": "number"}, "from_unit": {"type": "string"}, "to_unit": {"type": "string"}},
         "required": ["value", "from_unit", "to_unit"]}},
    {"name": "web_search",
     "description": "Searches DuckDuckGo for real-time facts.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "get_weather",
     "description": "Returns current weather for any city.",
     "input_schema": {"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]}},
    {"name": "run_python",
     "description": "Executes a Python code snippet and returns stdout. Always use print() to output results.",
     "input_schema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
]

print(f"Tools loaded: {[t['name'] for t in TOOL_SCHEMAS]}")

Tools loaded: ['calculator', 'get_current_time', 'lookup', 'unit_converter', 'web_search', 'get_weather', 'run_python']


## Step 4 — Define the ReAct agent

In [4]:
import anthropic
import time
from collections import defaultdict

MODEL      = "claude-sonnet-4-6"
MAX_STEPS  = 10
W          = 62
DIVIDER    = "─" * W

SYSTEM_PROMPT = """\
You are a ReAct (Reasoning + Acting) agent with access to a suite of powerful tools.

For every user question, follow this exact loop:
1. THINK  — Reason step by step. What do you need to find out or compute?
2. ACT    — Call the most appropriate tool. NEVER guess numbers, dates, or live facts.
3. OBSERVE — Examine the result. Is the question fully answered? If not, repeat.
4. ANSWER — Once fully informed, give a clear, accurate, complete final answer.

Tool selection guide:
  calculator      → arithmetic, powers, roots, trig, logarithms
  get_current_time → current date and time
  lookup          → definitions of AI/tech concepts
  unit_converter  → temperature, distance, weight conversions
  web_search      → real-time facts or anything outside the knowledge base
  get_weather     → live weather for any city
  run_python      → complex algorithms, list/string processing

Always use a tool for math — never compute in your head.
"""

def run_react(query: str, verbose: bool = True) -> str:
    """Run the ReAct loop for a single query and return the final answer."""
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": query}]
    tool_stats: dict[str, int] = defaultdict(int)
    start = time.time()

    print(f"\n{'═'*W}")
    print(f"QUERY: {query}")
    print(f"{'═'*W}")

    final_answer = ""

    for step in range(1, MAX_STEPS + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=16000,
            thinking={"type": "enabled", "budget_tokens": 8000},
            system=SYSTEM_PROMPT,
            tools=TOOL_SCHEMAS,
            messages=messages,
        )

        thought_text, text_output, tool_calls, assistant_blocks = "", "", [], []
        for block in response.content:
            assistant_blocks.append(block)
            if block.type == "thinking":  thought_text += block.thinking
            elif block.type == "text":    text_output  += block.text
            elif block.type == "tool_use": tool_calls.append(block)

        if verbose and thought_text:
            preview = thought_text.strip()[:300]
            if len(thought_text) > 300: preview += f" ... [{len(thought_text)-300} more chars]"
            print(f"\n[Step {step}]  THOUGHT\n{DIVIDER}\n{preview}")

        if not tool_calls:
            final_answer = text_output.strip()
            print(f"\n{'═'*W}\nFINAL ANSWER\n{DIVIDER}\n{final_answer}\n{'═'*W}")
            break

        messages.append({"role": "assistant", "content": assistant_blocks})

        tool_results = []
        for tc in tool_calls:
            tool_stats[tc.name] += 1
            print(f"\n[Step {step}]  ACTION\n{DIVIDER}")
            print(f"Tool  : {tc.name}")
            print(f"Input : {json.dumps(tc.input)}")

            result = execute_tool(tc.name, tc.input)

            print(f"\n[Step {step}]  OBSERVATION\n{DIVIDER}\n{result}")
            tool_results.append({"type": "tool_result", "tool_use_id": tc.id, "content": result})

        messages.append({"role": "user", "content": tool_results})

    else:
        final_answer = "[Max steps reached without a final answer]"
        print(f"\n[WARNING] {final_answer}")

    elapsed = time.time() - start
    tools_summary = ", ".join(f"{k}×{v}" for k, v in sorted(tool_stats.items())) or "none"
    status = "✓ Success" if final_answer and not final_answer.startswith("[") else "✗ Incomplete"

    print(f"\n╔{'═'*W}╗")
    print(f"║ {'EXECUTION SUMMARY':^{W-2}} ║")
    print(f"╠{'═'*W}╣")
    print(f"║   Steps taken  : {step:<{W-18}} ║")
    print(f"║   Tools called : {tools_summary:<{W-18}} ║")
    print(f"║   Elapsed time : {elapsed:.2f}s{'':<{W-22}} ║")
    print(f"║   Status       : {status:<{W-18}} ║")
    print(f"╚{'═'*W}╝")

    return final_answer

print("ReAct agent defined. Ready to run.")

ReAct agent defined. Ready to run.


## Step 5 — Run demo queries

Run each cell below to see the agent reason and act.

In [5]:
# Query 1: Calculator
run_react("What is 2 raised to the power of 10?");


══════════════════════════════════════════════════════════════
QUERY: What is 2 raised to the power of 10?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants to calculate 2 raised to the power of 10.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : calculator
Input : {"expression": "2**10"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
1024

══════════════════════════════════════════════════════════════
FINAL ANSWER
──────────────────────────────────────────────────────────────
**OBSERVE** — The calculator returned 1024. That's the complete answer.

**ANSWER** — 2 raised to the power of 10 is **1,024**! 🎯

In other words, 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 = **1,024**. This is also a well-known value in computing, as it equals **1 Kilobyte (1 KB)** in binary terms.
═══════════════════════════════

In [6]:
# Query 2: Current time
run_react("What is the current date and time?");


══════════════════════════════════════════════════════════════
QUERY: What is the current date and time?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants to know the current date and time. I'll use the get_current_time tool.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : get_current_time
Input : {}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
Thursday, May 07, 2026 at 09:06:20 AM

══════════════════════════════════════════════════════════════
FINAL ANSWER
──────────────────────────────────────────────────────────────
**OBSERVE:** The tool returned the current date and time successfully.

**ANSWER:** The current date and time is:

📅 **Thursday, May 7, 2026**
🕘 **9:06:20 AM**
══════════════════════════════════════════════════════════════

╔═════════════════════════════════════════════════════════

In [7]:
# Query 3: Multi-tool — calculator + lookup
run_react("What is the square root of 256, and what does 'ReAct' mean in AI?");


══════════════════════════════════════════════════════════════
QUERY: What is the square root of 256, and what does 'ReAct' mean in AI?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user asks two independent questions: the square root of 256 (math) and what 'ReAct' means in AI (lookup). I can call both tools simultaneously.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : calculator
Input : {"expression": "sqrt(256)"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
16.0

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : lookup
Input : {"term": "react"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
ReAct (Reasoning + Acting) is an AI agent paradigm that interleaves step-by-step reasoning (Thought) with tool invocations (Action) and

In [8]:
# Query 4: Multi-tool — unit converter + lookup
run_react("Convert 100 km to miles, and tell me what Anthropic is.");


══════════════════════════════════════════════════════════════
QUERY: Convert 100 km to miles, and tell me what Anthropic is.
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants two things: convert 100 km to miles, and get a definition of Anthropic. These are independent, so I can call both tools simultaneously.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : unit_converter
Input : {"value": 100, "from_unit": "km", "to_unit": "miles"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
100 km = 62.1371 miles

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : lookup
Input : {"term": "anthropic"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
Anthropic is an AI safety company founded in 2021 that created the Claude family of large 

In [9]:
# Query 5: Multi-tool — calculator + unit converter
run_react("Calculate (15 * 7) + sqrt(81), then convert 37 degrees Celsius to Fahrenheit.");


══════════════════════════════════════════════════════════════
QUERY: Calculate (15 * 7) + sqrt(81), then convert 37 degrees Celsius to Fahrenheit.
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants two things:
1. Calculate (15 * 7) + sqrt(81)
2. Convert 37 degrees Celsius to Fahrenheit

These are independent, so I can call both tools simultaneously.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : calculator
Input : {"expression": "(15 * 7) + sqrt(81)"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
114.0

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : unit_converter
Input : {"value": 37, "from_unit": "celsius", "to_unit": "fahrenheit"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
37 celsius = 98.6000 fahrenheit

═════

In [10]:
# Query 6: Web search
run_react("Search the web: what is Claude AI?");


══════════════════════════════════════════════════════════════
QUERY: Search the web: what is Claude AI?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants to search the web for information about Claude AI.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : web_search
Input : {"query": "what is Claude AI?"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
Claude is a series of large language models developed by Anthropic and first released in 2023. Since Claude 3, each generation has typically been released in three sizes, from least to most capable: Haiku, Sonnet, and Opus. An additional model named Claude Mythos was released to a handful of companies in 2026 but not to the public. Claude is used for software development via Claude Code. Claude is trained using "constitutional AI", a technique developed

In [11]:
# Query 7: Live weather
run_react("What's the weather like in London right now?");


══════════════════════════════════════════════════════════════
QUERY: What's the weather like in London right now?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants to know the current weather in London. I'll use the get_weather tool.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : get_weather
Input : {"location": "London"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
Weather for Westminster, United Kingdom:
  Condition  : Partly cloudy
  Temp       : 12°C / 54°F
  Humidity   : 71%
  Wind       : 5 km/h

══════════════════════════════════════════════════════════════
FINAL ANSWER
──────────────────────────────────────────────────────────────
Here's the current weather in **London, UK** 🇬🇧:

| Detail | Value |
|---|---|
| ☁️ **Condition** | Partly Cloudy |
| 🌡️ **Temperature** | 12°C / 54°F |
| 💧 *

In [12]:
# Query 8: Run Python code
run_react("Write and run Python code to generate the first 10 Fibonacci numbers.");


══════════════════════════════════════════════════════════════
QUERY: Write and run Python code to generate the first 10 Fibonacci numbers.
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants to generate the first 10 Fibonacci numbers using Python code.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : run_python
Input : {"code": "\ndef fibonacci(n):\n    fib_sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_sequence.append(a)\n        a, b = b, a + b\n    return fib_sequence\n\nfirst_10 = fibonacci(10)\nprint(\"First 10 Fibonacci Numbers:\")\nfor i, num in enumerate(first_10, 1):\n    print(f\"  F({i}) = {num}\")\n"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
First 10 Fibonacci Numbers:
  F(1) = 0
  F(2) = 1
  F(3) = 1
  F(4) = 2
  F(5) = 3
  F(6) = 5
  F(7) = 8
  F(8) = 13
  F(9

## Custom query

Try your own question:

In [13]:
run_react("What is the square root of 144, and what is today's date?");


══════════════════════════════════════════════════════════════
QUERY: What is the square root of 144, and what is today's date?
══════════════════════════════════════════════════════════════

[Step 1]  THOUGHT
──────────────────────────────────────────────────────────────
The user wants two things: the square root of 144 and today's date. These are independent, so I can call both tools simultaneously.

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : calculator
Input : {"expression": "sqrt(144)"}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
12.0

[Step 1]  ACTION
──────────────────────────────────────────────────────────────
Tool  : get_current_time
Input : {}

[Step 1]  OBSERVATION
──────────────────────────────────────────────────────────────
Thursday, May 07, 2026 at 09:07:23 AM

══════════════════════════════════════════════════════════════
FINAL ANSWER
─────────────────────────────────────────────────